In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

key = os.environ.get("GROQ_API_KEY", "")
print(f"Key starts with: {key[:8]}...")
print(f"Key length: {len(key)} chars")
print("✅ Ready!" if key.startswith("gsk_") and len(key) > 20 else "❌ Check your key!")

In [ ]:
!pip install mcp groq httpx duckduckgo-search==6.3.7 nest_asyncio -q
print("✅ All packages installed!")

In [ ]:
server_code = '''
import asyncio
import os
import requests
from groq import Groq
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
from concurrent.futures import ThreadPoolExecutor

app = Server("groq-research-assistant")
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.3-70b-versatile"
executor = ThreadPoolExecutor(max_workers=2)

def fetch_news_sync(query: str) -> list:
    results = []
    try:
        resp = requests.get("https://hacker-news.firebaseio.com/v0/topstories.json", timeout=5)
        ids = resp.json()[:15]
        for story_id in ids:
            story = requests.get(
                f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json", timeout=5
            ).json()
            if story and story.get("title"):
                results.append({
                    "title": story.get("title", ""),
                    "body": f"Score: {story.get('score', 0)} points, {story.get('descendants', 0)} comments",
                    "href": story.get("url") or f"https://news.ycombinator.com/item?id={story_id}"
                })
            if len(results) >= 5:
                break
    except Exception as e:
        results = [{"title": f"Error: {e}", "body": "", "href": ""}]
    return results

@app.list_tools()
async def list_tools() -> list[types.Tool]:
    return [
        types.Tool(
            name="summarize_text",
            description="Summarize any text into 5 clear bullet points",
            inputSchema={
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Text to summarize"}
                },
                "required": ["text"]
            }
        ),
        types.Tool(
            name="ask_groq",
            description="Answer any general knowledge question using Groq AI",
            inputSchema={
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "Question to answer"}
                },
                "required": ["question"]
            }
        ),
        types.Tool(
            name="search_web",
            description="Get live top tech and AI news headlines from the web",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Topic to search for"}
                },
                "required": ["query"]
            }
        ),
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:

    if name == "summarize_text":
        result = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Summarize as 5 clear bullet points."},
                {"role": "user", "content": arguments["text"]}
            ]
        )
        return [types.TextContent(type="text", text=result.choices[0].message.content)]

    elif name == "ask_groq":
        result = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Answer clearly and concisely."},
                {"role": "user", "content": arguments["question"]}
            ]
        )
        return [types.TextContent(type="text", text=result.choices[0].message.content)]

    elif name == "search_web":
        query = arguments.get("query", "tech news")
        loop = asyncio.get_event_loop()
        results = await loop.run_in_executor(executor, fetch_news_sync, query)

        raw = f"Top stories for: {query}\\n\\n"
        for i, r in enumerate(results, 1):
            raw += f"{i}. {r[\'title\']}\\n   {r[\'body\']}\\n   {r[\'href\']}\\n\\n"

        synthesis = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Summarize these headlines into a clear news briefing with source links at the end."},
                {"role": "user", "content": raw}
            ]
        )
        answer = synthesis.choices[0].message.content
        answer += "\\n\\n---\\n📚 Sources:\\n"
        for i, r in enumerate(results, 1):
            answer += f"{i}. {r[\'title\']} — {r[\'href\']}\\n"

        return [types.TextContent(type="text", text=answer)]

    return [types.TextContent(type="text", text=f"Unknown tool: {name}")]

async def main():
    async with stdio_server() as (read_stream, write_stream):
        await app.run(read_stream, write_stream, app.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("mcp_server.py", "w") as f:
    f.write(server_code)
print("✅ mcp_server.py created!")

In [ ]:
client_code = '''
import asyncio, os, json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
TOOL_MODEL  = "llama-3.1-8b-instant"
FINAL_MODEL = "llama-3.3-70b-versatile"
SEARCH_KEYWORDS = ["latest", "today", "news", "current", "recent", "now", "trending"]

async def run_agent(user_query: str):
    server_params = StdioServerParameters(
        command="python",
        args=["mcp_server.py"],
        env={"GROQ_API_KEY": os.environ["GROQ_API_KEY"]}
    )
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            tools = tools_result.tools
            print(f"🔧 Tools: {[t.name for t in tools]}")

            groq_tools = [
                {"type": "function", "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.inputSchema
                }} for t in tools
            ]

            needs_search = any(k in user_query.lower() for k in SEARCH_KEYWORDS)
            print(f"🔎 Search mode: {needs_search}")
            print(f"🤔 Query: {user_query}\\n")

            tool_choice = (
                {"type": "function", "function": {"name": "search_web"}}
                if needs_search else "auto"
            )

            messages = [
                {"role": "system", "content": "You are a research assistant. Use tools to answer. Return tool results directly without filler phrases."},
                {"role": "user", "content": user_query}
            ]

            last_tool_output = None

            for step in range(4):
                try:
                    response = groq_client.chat.completions.create(
                        model=TOOL_MODEL,
                        messages=messages,
                        tools=groq_tools,
                        tool_choice=tool_choice
                    )
                except Exception as e:
                    print(f"⚠️ Error: {e}")
                    fallback = groq_client.chat.completions.create(
                        model=FINAL_MODEL,
                        messages=[{"role":"system","content":"Answer clearly."},{"role":"user","content":user_query}]
                    )
                    print("\\n🎯 Final Answer:\\n")
                    print(fallback.choices[0].message.content)
                    return

                msg = response.choices[0].message
                tool_choice = "auto"  # reset after first call

                if not msg.tool_calls:
                    final = msg.content or last_tool_output or "No answer generated."
                    filler = ["i hope","if you have","don't hesitate","feel free","no further","please note"]
                    if any(p in final.lower() for p in filler) and last_tool_output:
                        final = last_tool_output
                    print("\\n🎯 Final Answer:\\n")
                    print(final)
                    return

                messages.append({
                    "role": "assistant",
                    "content": msg.content or "",
                    "tool_calls": [
                        {"id": tc.id, "type": "function",
                         "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                        for tc in msg.tool_calls
                    ]
                })

                for tc in msg.tool_calls:
                    tool_name = tc.function.name
                    try:
                        tool_args = json.loads(tc.function.arguments)
                    except:
                        tool_args = (
                            {"query": user_query} if tool_name == "search_web" else
                            {"question": user_query} if tool_name == "ask_groq" else
                            {"text": user_query}
                        )

                    print(f"⚙️  Tool: {tool_name} | Args: {tool_args}")
                    result = await session.call_tool(tool_name, tool_args)
                    last_tool_output = result.content[0].text if result.content else "No result"
                    print(f"📤 Preview: {last_tool_output[:120]}...\\n")

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": last_tool_output
                    })

            if last_tool_output:
                print("\\n🎯 Final Answer:\\n")
                print(last_tool_output)

if __name__ == "__main__":
    import sys
    query = sys.argv[1] if len(sys.argv) > 1 else "What are the latest AI news today?"
    asyncio.run(run_agent(query))
'''

with open("mcp_client.py", "w") as f:
    f.write(client_code)
print("✅ mcp_client.py created!")

✅ mcp_client.py created!


In [ ]:
import subprocess, os

queries = [
    "What are the latest AI news today?",
    "Summarize what transformers are in ML",
    "What is the difference between RAM and ROM?",
]

for q in queries:
    print(f"\n{'='*60}\n📌 {q}\n{'='*60}")
    result = subprocess.run(
        ["python", "mcp_client.py", q],
        capture_output=True, text=True,
        env={**os.environ}
    )
    print(result.stdout)
    if result.returncode != 0:
        print("⚠️ Error:", result.stderr[-400:])

In [ ]:
# Cell 6 (Fixed): Run queries by passing them as CLI args — no file rewriting needed

import subprocess, os

queries = [
    "Summarize what quantum computing is in bullet points",
    "Explain RAG (Retrieval Augmented Generation) simply",
    "What are the key differences between SQL and NoSQL?",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"📌 {q}")
    print('='*60)

    result = subprocess.run(
        ["python", "mcp_client.py", q],   # ✅ pass query as arg, no temp file
        capture_output=True,
        text=True,
        env={**os.environ}
    )

    print(result.stdout)
    if result.returncode != 0:
        print("⚠️ Error:", result.stderr[-800:])